# 05 — Bases finales para modelar nivel de riesgo

Este cuaderno crea dos conjuntos sin nulos en las variables incluidas y con corte temporal fijo. No entrena algoritmos: deja las bases listas para la evaluación posterior.

- **Evaluación inicial:** variables demográficas y relacionales de cobertura completa, excluyendo geografía/CEM, condición administrativa y acciones posteriores.
- **Réplica retrospectiva:** añade `TIPO_VIOLENCIA`, conocido en el registro de atención, para estudiar la reproducibilidad de la valoración registrada. No representa una predicción previa a la valoración.

## Sustento de las decisiones

La selección anterior se audita en el notebook 04. Aquí se parte de variables obligatorias/sin nulos y se preserva la separación temporal: entrenamiento 2020–2023, validación 2024 y prueba final 2025. Ningún cálculo de imputación, codificación o selección debe aprender de los conjuntos de validación/prueba.

In [1]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'salidas_limpieza' / 'base_historica_2020_2025_ubigeo.parquet'
AUDIT_PATH = BASE_DIR / 'salidas_modelado' / 'auditoria_top30_previa_nivel_riesgo.csv'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado' / 'bases_finales_riesgo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATE_FORMAT = '%m/%d/%Y'
EJECUTAR_EXPORTACION = True  # Cambiar a True después de revisar la ficha de variables.

assert DATA_PATH.exists(), f'No existe: {DATA_PATH}'
assert AUDIT_PATH.exists(), 'Ejecute el notebook 04 antes de este.'
df = pd.read_parquet(DATA_PATH)
df['anio_ingreso'] = pd.to_datetime(df['FECHA_INGRESO'], format=DATE_FORMAT, errors='raise').dt.year
print(f'Registros disponibles: {len(df):,}')

Registros disponibles: 936,835


## 1. Especificación aprobada de variables

Se excluyen por diseño `CEM`, UBIGEO y códigos geográficos (contexto/fairness, no predictores individuales por defecto); `CONDICION` (posible fuga administrativa); y decisiones posteriores como patrocinio legal o deseo de denunciar. La versión retrospectiva agrega el tipo de violencia con alcance explícitamente descriptivo-retrospectivo.

In [2]:
TARGET = 'NIVEL_DE_RIESGO_VICTIMA'
TARGET_LABELS = {'1': 'Leve', '2': 'Moderado', '3': 'Severo'}

FEATURES_INICIAL = [
    'EDAD_VICTIMA', 'SEXO_VICTIMA', 'VICTIMA_PERUANA', 'VICTIMA_EXTRANJERA',
    'AREA_RESIDENCIA_DOMICILIO', 'ESTADO_CIVIL_VICTIMA', 'NIVEL_EDUCATIVO_VICTIMA',
    'ESTUDIA', 'TRABAJA_VICTIMA', 'VINCULO_AGRESOR_VICTIMA',
    'AGRESOR_VIVE_CASA_VICTIMA', 'SEXO_AGRESOR', 'DEPENDE_VICTIMA_FEMINICIDIO',
    'PRIMERA_VEZ_AGREDE', 'ESTADO_AGRESOR_U_A', 'ESTADO_VICTIMA_U_A', 'REDES_FAM_SOC',
]
FEATURES_RETROSPECTIVA = [*FEATURES_INICIAL, 'TIPO_VIOLENCIA']

for nombre, features in {'inicial': FEATURES_INICIAL, 'retrospectiva': FEATURES_RETROSPECTIVA}.items():
    columnas = [*features, TARGET]
    assert set(columnas).issubset(df.columns), f'Faltan columnas en escenario {nombre}'
    nulos = df[columnas].isna().sum()
    assert int(nulos.sum()) == 0, f'El escenario {nombre} no es completo: {nulos[nulos.gt(0)].to_dict()}'
    print(f'{nombre}: {len(features)} predictores y 0 nulos en {len(columnas)} columnas.')

ficha = pd.DataFrame([
    *[['inicial', x, 'predictor', 'Cobertura completa; requiere codificación aprendida solo con train.'] for x in FEATURES_INICIAL],
    ['inicial', TARGET, 'target', 'Etiqueta registrada: 1 Leve, 2 Moderado, 3 Severo.'],
    ['retrospectiva', 'TIPO_VIOLENCIA', 'predictor adicional', 'Solo para reproducir retrospectivamente la valoración; no para predicción previa.'],
], columns=['escenario', 'variable', 'rol', 'justificacion'])
display(ficha)

inicial: 17 predictores y 0 nulos en 18 columnas.
retrospectiva: 18 predictores y 0 nulos en 19 columnas.


,escenario,variable,rol,justificacion
0,inicial,EDAD_VICTIMA,predictor,Cobertura completa; requiere codificación apre...
1,inicial,SEXO_VICTIMA,predictor,Cobertura completa; requiere codificación apre...
2,inicial,VICTIMA_PERUANA,predictor,Cobertura completa; requiere codificación apre...
3,inicial,VICTIMA_EXTRANJERA,predictor,Cobertura completa; requiere codificación apre...
4,inicial,AREA_RESIDENCIA_DOMICILIO,predictor,Cobertura completa; requiere codificación apre...
5,inicial,ESTADO_CIVIL_VICTIMA,predictor,Cobertura completa; requiere codificación apre...
6,inicial,NIVEL_EDUCATIVO_VICTIMA,predictor,Cobertura completa; requiere codificación apre...
7,inicial,ESTUDIA,predictor,Cobertura completa; requiere codificación apre...
8,inicial,TRABAJA_VICTIMA,predictor,Cobertura completa; requiere codificación apre...
9,inicial,VINCULO_AGRESOR_VICTIMA,predictor,Cobertura completa; requiere codificación apre...


## 2. Diagnóstico previo a exportar

Las clases objetivo se conservan completas. El corte por año es fijo y no aleatorio para detectar cambios de formulario, composición o práctica de registro.

In [3]:
diagnostico = (df.groupby('anio_ingreso')[TARGET].value_counts(normalize=True).mul(100).rename('pct')
              .reset_index())
diagnostico['etiqueta'] = diagnostico[TARGET].astype('string').map(TARGET_LABELS)
display(diagnostico.pivot(index='anio_ingreso', columns='etiqueta', values='pct').round(2))

splits = {
    'train_2020_2023': [2020, 2021, 2022, 2023],
    'validacion_2024': [2024],
    'prueba_2025': [2025],
}
for nombre, anios in splits.items():
    print(f'{nombre}: {(df["anio_ingreso"].isin(anios)).sum():,} registros')

etiqueta,Leve,Moderado,Severo
anio_ingreso,,,
2020,25.56,50.79,23.64
2021,24.93,51.93,23.14
2022,22.53,53.15,24.31
2023,20.34,52.07,27.59
2024,18.76,51.55,29.69
2025,18.42,51.21,30.37


train_2020_2023: 598,807 registros
validacion_2024: 168,492 registros
prueba_2025: 169,536 registros


## 3. Exportación de los Parquet finales

Al activar esta celda se crean seis archivos (tres cortes por cada escenario) y una ficha de variables. Solo se sobrescriben archivos dentro de `salidas_modelado/bases_finales_riesgo/`.

In [4]:
if EJECUTAR_EXPORTACION:
    escenarios = {
        'inicial': FEATURES_INICIAL,
        'retrospectiva': FEATURES_RETROSPECTIVA,
    }
    resumen = []
    for escenario, features in escenarios.items():
        columnas = [*features, TARGET, 'anio_ingreso']
        base = df[columnas].copy()
        base['nivel_riesgo_etiqueta'] = base[TARGET].astype('string').map(TARGET_LABELS)
        assert int(base.isna().sum().sum()) == 0, f'Persisten nulos en {escenario}'
        for corte, anios in splits.items():
            salida = base.loc[base['anio_ingreso'].isin(anios)].copy()
            salida.to_parquet(OUTPUT_DIR / f'riesgo_{escenario}_{corte}.parquet', index=False)
            resumen.append({
                'escenario': escenario, 'corte': corte, 'anios': '-'.join(map(str, anios)),
                'filas': len(salida), 'columnas': len(salida.columns), 'nulos': int(salida.isna().sum().sum()),
            })
    ficha.to_csv(OUTPUT_DIR / 'ficha_variables_bases_finales.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(resumen).to_csv(OUTPUT_DIR / 'resumen_bases_finales.csv', index=False, encoding='utf-8-sig')
    display(pd.DataFrame(resumen))
else:
    print('Exportación no ejecutada. Cambie EJECUTAR_EXPORTACION a True para crear los Parquet finales.')

,escenario,corte,anios,filas,columnas,nulos
0,inicial,train_2020_2023,2020-2021-2022-2023,598807,20,0
1,inicial,validacion_2024,2024,168492,20,0
2,inicial,prueba_2025,2025,169536,20,0
3,retrospectiva,train_2020_2023,2020-2021-2022-2023,598807,21,0
4,retrospectiva,validacion_2024,2024,168492,21,0
5,retrospectiva,prueba_2025,2025,169536,21,0


## Próxima fase

El notebook de evaluación debe ajustar one-hot encoding, selección de características, balanceo y modelos únicamente sobre `train_2020_2023`. Usará 2024 para las decisiones y abrirá 2025 una sola vez para la estimación final. Se reportará macro-F1, precisión/recall por clase, matriz de confusión y calibración; no solo accuracy.